In [ ]:
import numpy as np 
import pandas as pd 

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [2]:
import PIL
import os
import numpy as np
import pandas as pd
import keras
import cv2
import seaborn as sns
import matplotlib.pyplot as plt
from keras.models import Sequential
from PIL import Image
from keras.layers import Conv2D,Flatten,Dense,Dropout,BatchNormalization,MaxPooling2D
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import label_binarize

from tensorflow.keras.utils import to_categorical
from keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from keras.callbacks import ReduceLROnPlateau, EarlyStopping

%matplotlib inline

In [ ]:
glioma="/kaggle/input/brain-tumour-classification/BrainTumor_1/Train/glioma"
image_files = os.listdir(glioma)
plt.figure(figsize=(10, 10))

for i in range(25):
	img_path = os.path.join(glioma, image_files[i])
	img = cv2.imread(img_path)
	img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
	plt.subplot(5, 5, i + 1)
	plt.imshow(img)
	plt.axis('off')
	
plt.suptitle('Glioma Samples', fontsize=18, fontweight='bold')
plt.tight_layout()
plt.savefig("glioma.jpg", dpi=1200, bbox_inches='tight')
plt.show()

In [ ]:
meningioma_dir="/kaggle/input/brain-tumour-classification/BrainTumor_1/Train/meningioma"
image_files = os.listdir(meningioma_dir)
plt.figure(figsize=(10, 10))
for i in range(25):
    img_path = os.path.join(meningioma_dir, image_files[i])
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.subplot(5, 5, i + 1)
    plt.imshow(img)
    plt.axis('off')

plt.suptitle('Meningioma Samples', fontsize=18, fontweight = 'bold')
plt.tight_layout()
plt.savefig("meningioma.jpg", dpi=1200, bbox_inches='tight')
plt.show()

In [ ]:
notumor_dir="/kaggle/input/brain-tumour-classification/BrainTumor_1/Train/notumor"
image_files = os.listdir(notumor_dir)
plt.figure(figsize=(10, 10))

for i in range(25):
	img_path = os.path.join(notumor_dir, image_files[i])
	img = cv2.imread(img_path)
	img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
	plt.subplot(5, 5, i + 1)
	plt.imshow(img)
	plt.axis('off')
    
plt.suptitle('No Tumor/Safe Samples', fontsize=18, fontweight = 'bold')
plt.tight_layout()
plt.savefig("notumor.jpg", dpi=1200, bbox_inches='tight')
plt.show()

In [ ]:
pituitary_dir="/kaggle/input/brain-tumour-classification/BrainTumor_1/Train/pituitary"
image_files = os.listdir(pituitary_dir)
plt.figure(figsize=(10, 10))
for i in range(25):
	img_path = os.path.join(pituitary_dir, image_files[i])
	img = cv2.imread(img_path)
	img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
	plt.subplot(5, 5, i + 1)
	plt.imshow(img)
	plt.axis('off')
    
plt.suptitle('Pituitary Tumor Samples', fontsize=18, fontweight = 'bold')
plt.tight_layout()
plt.savefig("pituitary.jpg", dpi=1200, bbox_inches='tight')
plt.show()

In [9]:
image_size=[150,150]
train_data="/kaggle/input/brain-tumour-classification/BrainTumor_1/Train"
test_data="/kaggle/input/brain-tumour-classification/BrainTumor_1/Test"
valid_data="/kaggle/input/brain-tumour-classification/BrainTumor_1/Test"

In [ ]:
filepath=[]
label=[]
image_folder=os.listdir(train_data)
for folder in image_folder:
    folder_path=os.path.join(train_data,folder)
    filelist=os.listdir(folder_path)
    for file in filelist:
        new_path=os.path.join(folder_path,file)
        filepath.append(new_path)
        label.append(folder)
image_data=pd.Series(filepath,name="image_data")
label_data=pd.Series(label,name="label")
train_df=pd.concat([image_data,label_data],axis=1)
train_df.head()

In [ ]:
train_df.shape

In [ ]:
train_df.isnull().sum()

In [ ]:
train_df["label"].value_counts()

In [ ]:
labels = train_df["label"].value_counts().index
sizes = train_df["label"].value_counts().values
colors = ['#E35675', '#9556E3', '#E36656', '#674EE6']
fig, ax = plt.subplots(figsize=(8, 8))
wedges, texts, autotexts = ax.pie(sizes, labels=labels, colors=colors,autopct='%1.1f%%',startangle=140, wedgeprops={'edgecolor':'white'})
centre_circle = plt.Circle((0,0), 0.70, fc='white')
fig.gca().add_artist(centre_circle)
ax.axis('equal')
plt.show()

In [ ]:
filepath=[]
label=[]
image_folder=os.listdir(test_data)
for folder in image_folder:
    folder_path=os.path.join(test_data,folder)
    filelist=os.listdir(folder_path)
    for file in filelist:
        new_path=os.path.join(folder_path,file)
        filepath.append(new_path)
        label.append(folder)
image_data=pd.Series(filepath,name="image_data")
label_data=pd.Series(label,name="label")
test_df=pd.concat([image_data,label_data],axis=1)
test_df.head()

In [ ]:
test_df.shape

In [ ]:
test_df.isnull().sum()

In [ ]:
test_df["label"].value_counts()

In [ ]:
label_counts = test_df["label"].value_counts()
plt.figure(figsize=(8, 8))

color=["#4E66E6","green","crimson","darkorange"]
plt.pie(label_counts, labels=label_counts.index, autopct='%1.1f%%',startangle=140,colors=color)
plt.axis('equal')
plt.show()

In [ ]:
filepath=[]
label=[]
image_folder=os.listdir(valid_data)
for folder in image_folder:
    folder_path=os.path.join(valid_data,folder)
    filelist=os.listdir(folder_path)
    for file in filelist:
        new_path=os.path.join(folder_path,file)
        filepath.append(new_path)
        label.append(folder)

image_data=pd.Series(filepath,name="image_data")
label_data=pd.Series(label,name="label")
valid_df=pd.concat([image_data,label_data],axis=1)
valid_df.head()

In [ ]:
valid_df.shape

In [ ]:
valid_df["label"].value_counts()

In [ ]:
plt.figure(figsize=(10,8))
sns.countplot(data=valid_df,x="label",palette="tab10")
plt.show()

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

batch_size=16
color_channel=3
image_shape=(150,150,3)

train_generator = ImageDataGenerator(
rescale=1./255,
rotation_range=20,
width_shift_range=0.1,
height_shift_range=0.1,
shear_range=0.2,
zoom_range=0.2,
horizontal_flip=True,
fill_mode='nearest')

test_generator=ImageDataGenerator(rescale=1./255)


train_data_generator=train_generator.flow_from_dataframe(train_df,x_col="image_data",y_col="label",target_size=image_size,
color_mode="rgb",
shuffle=True,
batch_size=batch_size)

test_data_generator=test_generator.flow_from_dataframe(test_df,x_col="image_data",y_col="label",
target_size=image_size,
color_mode="rgb",
shuffle=False,
batch_size=batch_size)

valid_data_generator=test_generator.flow_from_dataframe(valid_df,x_col="image_data",y_col="label",
target_size=image_size,
color_mode="rgb",
shuffle=True,
batch_size=batch_size)

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import DenseNet169
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam


batch_size = 16
image_shape = (150, 150, 3)

In [ ]:
# Function to create DenseNet-169 model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import Precision, Recall, AUC

def create_densenet169_model(input_shape, num_classes):
    base_model = DenseNet169(weights='imagenet', include_top=False,input_shape=input_shape)
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(1024, activation='relu')(x)
    predictions = Dense(num_classes, activation='softmax')(x)
    model = Model(inputs=base_model.input, outputs=predictions)
    for layer in base_model.layers:
        layer.trainable = False
    return model

num_classes = len(train_data_generator.class_indices)
model = create_densenet169_model(image_shape, num_classes)
model.compile(optimizer=Adam(learning_rate=0.001),loss='categorical_crossentropy', metrics=['accuracy', 'mse', Precision(), Recall(), AUC()])
model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
batch_size = 4
epochs = 200

early_stopping = EarlyStopping(monitor = 'val_loss', min_delta = 0, patience = 15, verbose = 1, mode = 'auto',restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10,min_lr=1e-6)
history = model.fit(train_data_generator, 
                    batch_size = batch_size, 
                    validation_data=valid_data_generator,
                    epochs = epochs,
                    callbacks=[early_stopping, reduce_lr], verbose = 1)

In [ ]:
## Usable graphs
plt.figure(figsize=(15,6), dpi = 300)
# Plotting the training and validation loss
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss',color='#183B59',marker="*",lw=3,markersize=8,linestyle="--")
plt.plot(history.history['val_loss'], label='Validation Loss',color='#5C8C46',marker="P",lw=3,markersize=8,linestyle="--")
plt.legend()
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Loss' , fontsize = 14, fontweight ='bold')
plt.grid(linestyle = '--', linewidth = 0.5)
# Plotting the training and validation accuracy
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Accuracy',color='#274659',marker="D",lw=3,markersize=8,linestyle=":")
plt.plot(history.history['val_accuracy'], label='Validation Accuracy',color='#BFB3A8',marker="H",lw=3,markersize=8,linestyle=":")
plt.legend()
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Accuracy', fontsize = 14, fontweight ='bold')
plt.grid(linestyle = '--', linewidth = 0.5)
plt.savefig("acc1.pdf", dpi = 1200, 
            bbox_inches ="tight") 
plt.show()

In [ ]:
# Plotting accuracy vs epoch
plt.figure(figsize=(12, 4), dpi = 300)
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy vs. Epochs', fontsize = 14, fontweight ='bold')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# Plotting loss vs epoch
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss vs. Epochs', fontsize = 14, fontweight ='bold')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.savefig("acc2.pdf", dpi = 1200, 
            bbox_inches ="tight") 
plt.show()

In [ ]:
# Plot Training Accuracy and Loss in a single plot
plt.figure(figsize=(10, 6))
plt.plot(history.history['accuracy'], label='Accuracy', color='#E37656',marker="D",lw=3,markersize=8,linestyle=":")
plt.plot(history.history['loss'], label='Loss', color='#E3A756', marker='*', lw=3, markersize=8, linestyle='--')
plt.legend()
plt.xlabel('Epochs')
plt.ylabel('Value')
plt.title('Accuracy and Loss Plot Vs Epochs', fontsize=14, fontweight='bold')
plt.grid(linestyle='--', linewidth=0.5)
plt.savefig("train_accloss.pdf", dpi=1200, bbox_inches='tight')
plt.show()

# Plot Training Precision vs. Epoch
plt.figure(figsize=(10, 6))
plt.plot(history.history['precision'], label='Precision', color='#5669E3', marker='o', lw=3, markersize=8, linestyle='-')
plt.legend()
plt.xlabel('Epochs')
plt.ylabel('Precision')
plt.title('Precision Vs Epochs', fontsize=14, fontweight='bold')
plt.grid(linestyle='--', linewidth=0.5)
plt.savefig("train_precision.pdf", dpi=1200, bbox_inches='tight')
plt.show()

# Plot Training Recall vs. Epoch
plt.figure(figsize=(10, 6))
plt.plot(history.history['recall'], label='Recall', color='#6f81f2', marker='s', lw=3, markersize=8, linestyle='-')
plt.legend()
plt.xlabel('Epochs')
plt.ylabel('Recall')
plt.title('Recall Vs Epochs', fontsize=14, fontweight='bold')
plt.grid(linestyle='--', linewidth=0.5)
plt.savefig("train_recall.pdf", dpi=1200, bbox_inches='tight')
plt.show()

In [34]:
# Save the entire model to a HDF5 file
model.save('dense.h5')

In [ ]:
test_loss, test_acc, test_auc, test_mse, test_precision, test_recall = model.evaluate(test_data_generator, verbose=1)
print(f'Test Accuracy: {test_acc:.6f}')
print(f'Test Loss: {test_loss:.6f}')
print(f'Test AUC: {test_auc:.6f}')
print(f'Test MSE: {test_mse:.6f}')
print(f'Test Precision: {test_precision:.6f}')
print(f'Test Recall: {test_recall:.6f}')

In [ ]:
test_data_generator.class_indices

In [37]:
label_name=["glioma","meningioma","notumor","pituitary"]

In [ ]:
pred=model.predict(test_data_generator)
final=np.argmax(pred,axis=1)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, roc_curve,roc_auc_score
from sklearn.metrics import log_loss, cohen_kappa_score, matthews_corrcoef,brier_score_loss,accuracy_score
true_data=test_data_generator.classes
cf=confusion_matrix(test_data_generator.classes,final)
plt.figure(figsize=(15,12), dpi=300)
plt.rcParams.update({'font.size': 18})

sns.heatmap(cf, annot=True,fmt="d", cmap="GnBu", linewidth=.5, square=True, xticklabels=label_name,yticklabels=label_name)
plt.title('Confusion Matrix', fontsize = 14, fontweight ='bold')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.savefig("cm.pdf", dpi = 1200, 
            bbox_inches ="tight")
plt.show()

In [ ]:
print(classification_report(true_data,final,target_names=label_name))

In [ ]:
y_true = test_data_generator.classes
y_pred_prob = model.predict(test_data_generator)
classes = {'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}
y_true_binarized = label_binarize(y_true, classes=range(len(classes)))
fpr = dict()
tpr = dict()
roc_auc = dict()
for i in range(len(classes)):
    fpr[i], tpr[i], _ = roc_curve(y_true_binarized[:, i], y_pred_prob[:, i])
    roc_auc[i] = roc_auc_score(y_true_binarized[:, i], y_pred_prob[:, i])
plt.figure(figsize=(10, 8))
for i, label in enumerate(classes.keys()):
    plt.plot(fpr[i], tpr[i], label=f'{label} (AUC = {roc_auc[i]:.2f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic for Each Class')
plt.legend(loc='lower right')
plt.savefig("rocauc.pdf", dpi = 1200, 
            bbox_inches ="tight")
plt.show()

In [ ]:
acc_score = accuracy_score(y_true,final)
print(acc_score)

In [ ]:
roc_auc = roc_auc_score(y_true, pred,multi_class='ovr')
print(roc_auc)

In [ ]:
logarithm_loss=log_loss(y_true,pred)
print(logarithm_loss)

In [ ]:
kappa = cohen_kappa_score(y_true,final)
print(kappa)

In [ ]:
mcc = matthews_corrcoef(y_true, final)
print(mcc)

In [48]:
hist_df = pd.DataFrame(history.history)
hist_csv_file = 'data.csv'
with open(hist_csv_file, mode='w') as f:
    hist_df.to_csv(f)

In [ ]:
num_classes=4
true_data_one_hot = np.eye(num_classes)[true_data]
brier_scores = []
for i in range(num_classes):
    brier_scores.append(brier_score_loss(true_data_one_hot[:, i], pred[:, i]))
average_brier_score = np.mean(brier_scores)
plt.plot([])
plt.text(0,0, f' Average Brier Score Loss: {mcc:.4f}', fontsize=20,ha='center', va='center',color="crimson")
plt.axis('off')
plt.xlim(-1, 1)
plt.ylim(-1,1)
plt.show()

In [ ]:
class_name = list(test_data_generator.class_indices.keys())
plt.figure(figsize=(30,30))
number_images=(5,5)
for i in range(1,(number_images[0]*number_images[1])+1):
    plt.subplot(number_images[0],number_images[1],i)
    plt.axis("off")
    true_label = class_name[test_data_generator.classes[i]]
    predicted_label = class_name[final[i]]
    color="green"
    if true_label != predicted_label:
        color="red"
    plt.title(f"True: {true_label}\nPredicted: {predicted_label}", color=color)
    plt.imshow(plt.imread(test_data_generator.filepaths[i]))

plt.savefig("test.pdf", dpi = 1200, 
            bbox_inches ="tight")
plt.show()